# Practice 41 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: `.str.contains()` and `.str.extract()`

pandas `.str` methods let you work with string columns without loops or `.apply()`.

**`.str.contains(pattern)`** — returns a boolean Series: True where the string matches the pattern.
```python
df['name'].str.contains('ford')          # exact substring match
df['name'].str.contains('ford', case=False)   # case-insensitive
df['name'].str.contains(r'^toyota')      # regex: starts with 'toyota'
```

**`.str.extract(pattern)`** — pulls out the part of the string that matches a regex group `()`.
```python
# Extract digits from strings like 'Model-42', 'Item-7'
df['col'].str.extract(r'(\d+)')          # returns a DataFrame with one column
df['col'].str.extract(r'(\d+)')[0]       # get the Series

# Extract the word before a dash
df['col'].str.extract(r'^(\w+)-')        # captures word at the start before '-'
```

Both accept regular expressions. Key regex basics:
- `\d` — any digit, `\w` — any word character (letter/digit/underscore)
- `+` — one or more, `*` — zero or more
- `^` — start of string, `$` — end of string
- `( )` — capture group (required for `.str.extract()`)

---
### Task: mpg name column

Load `sns.load_dataset('mpg')`. The `name` column has values like `'chevrolet chevelle malibu'`, `'ford torino'`, `'toyota corolla'`.

1. Use `.str.contains()` to find all cars made by `'ford'` (case-insensitive). How many are there?
2. Use `.str.extract()` to pull out the **first word** of each car name as a new column `make`. (Hint: `r'^(\w+)'`)
3. Which `make` appears most often? Use `.value_counts()` — no groupby needed.
4. Use `.str.contains()` to flag cars whose name contains `'pickup'` or `'truck'` — one expression using `|` inside the regex pattern. How many?
5. Convert `make` to `category`. How much memory does it save vs `object` dtype?

In [44]:
# Your code here

mpg = sns.load_dataset('mpg')
mpg[mpg['name'].str.contains('ford', case=False)]
#mpg['name'].str.contains('ford', case=False).sum()

###

mpg['make'] = mpg['name'].str.extract(r'^(\w+)')

###
mpg['make'].value_counts().idxmax()


###
flgs = mpg['name'].str.contains(r'pickup|truck',na = False)
flgs.sum()

bm = mpg.memory_usage(deep= True).sum()
print(f'before {bm:.2f}')
mpg['make'] = mpg['make'].astype('category')
am = mpg.memory_usage(deep= True).sum()
print(f'after {am:.2f}')



before 100845.00
after 79493.00


---
## Level 2 — titanic: string + vectorized

Load `sns.load_dataset('titanic')`. No `.apply()`.

The `deck` column contains letter codes like `'A'`, `'B'`, `'C'` etc (with many nulls).

1. Use `.str.contains()` to flag passengers whose `deck` is in the upper decks (`'A'`, `'B'`, or `'C'`). Handle NaN with `na=False`.
2. The `embarked` column has codes `'S'`, `'C'`, `'Q'`. Use `.str.extract()` or `np.select()` to add a `port` column with full names: `'Southampton'`, `'Cherbourg'`, `'Queenstown'`.
3. Add `title` by extracting the title from the `name` column using `.str.extract()`. Titles look like `'Mr.'`, `'Mrs.'`, `'Miss.'`, `'Master.'` — pattern: `r'([A-Z][a-z]+\.)` 
4. Use `.str.contains()` to flag passengers with a rare title (not Mr, Mrs, Miss, or Master). What fraction of them survived?
5. Build a pivot table: mean `survived` by `title` × `pclass`, `observed=True`. Stack it and find the top combination.

In [ ]:
# Your code here

titanic = sns.load_dataset("titanic")

titanic['deck'].str.contains('A|B|C', na = False)

port_map  = {'S':'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
titanic['port'] = titanic['embarked'].map(port_map)

### there is no name column in this df 



In [32]:
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,port
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,Southampton
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,Southampton
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,Southampton
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,Southampton
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,Cherbourg


---
## Level 3 — pipeline with string ops

You have a product dataset below with messy string columns. Write a `.pipe()` chain — no `.apply()`.

- **`clean(df)`** — use `.str.extract()` to pull the numeric part from `sku` into a new column `sku_num` (as integer). Use `.str.contains()` to add `is_sale`: True where `tags` contains `'sale'` (case-insensitive). Convert `category` to `category` dtype.
- **`enrich(df)`** — add `final_price`: if `is_sale`, apply a 20% discount (price × 0.8), otherwise keep original price — vectorized using `np.where`. Add `margin_class` with `np.select()`: `'high'` (margin > 0.4), `'mid'` (margin > 0.2), `'low'` otherwise.

After the chain:
- What is the mean `final_price` per `category`? One groupby.
- Use a list comprehension with `zip` to collect product names where `is_sale` is True.
- Use `.str.contains()` to find products whose `name` contains a digit — any product names that look like model numbers?

In [47]:
products = pd.DataFrame({
    'name':     ['Widget Pro', 'Gadget X200', 'Doohickey', 'Thingamajig 3', 'Gizmo',
                 'Widget Lite', 'Gadget X100', 'Contraption', 'Widget Max', 'Doohickey Plus'],
    'sku':      ['WG-001', 'GD-002', 'DH-003', 'TM-004', 'GZ-005',
                 'WG-006', 'GD-007', 'CT-008', 'WG-009', 'DH-010'],
    'category': ['Widgets','Gadgets','Doohickeys','Thingamajigs','Gizmos',
                 'Widgets','Gadgets','Contraptions','Widgets','Doohickeys'],
    'price':    [29.99, 149.99, 9.99, 49.99, 19.99, 14.99, 99.99, 79.99, 59.99, 24.99],
    'margin':   [0.45, 0.30, 0.15, 0.38, 0.22, 0.50, 0.28, 0.18, 0.42, 0.20],
    'tags':     ['bestseller', 'sale,new', 'sale', 'new', 'bestseller',
                 'sale,bestseller', 'new', 'sale', 'bestseller', 'new']
})

# Your code here

def clean(df):
    df['sku_num'] = df['sku'].str.extract(r'(\d+)')[0].astype(int)
    df['is_sale'] = df['tags'].str.contains('sale',case = False)
    df['category'] = df['category'].astype('category')
    return df

def enrich(df):
    df['final_price'] = np.where(df['is_sale'], df['price']*0.8, df['price'])
    conds = [df['margin']>0.4, df['margin']>0.2]
    chos = ['high','mid']
    df['margin_class'] = np.select(conds, chos, default='low')
    return df 

res = products.copy().pipe(clean).pipe(enrich)


res.groupby('category', observed=True)['final_price'].mean()
[name for name, cat in zip(res['name'], res['is_sale']) if cat]
res['name'][res['name'].str.contains(r'\d')]


1      Gadget X200
3    Thingamajig 3
6      Gadget X100
Name: name, dtype: object

In [48]:
res

,name,sku,category,price,margin,tags,sku_num,is_sale,final_price,margin_class
0,Widget Pro,WG-001,Widgets,29.99,0.45,bestseller,1,False,29.990,high
1,Gadget X200,GD-002,Gadgets,149.99,0.30,"sale,new",2,True,119.992,mid
2,Doohickey,DH-003,Doohickeys,9.99,0.15,sale,3,True,7.992,low
3,Thingamajig 3,TM-004,Thingamajigs,49.99,0.38,new,4,False,49.990,mid
4,Gizmo,GZ-005,Gizmos,19.99,0.22,bestseller,5,False,19.990,mid
5,Widget Lite,WG-006,Widgets,14.99,0.50,"sale,bestseller",6,True,11.992,high
6,Gadget X100,GD-007,Gadgets,99.99,0.28,new,7,False,99.990,mid
7,Contraption,CT-008,Contraptions,79.99,0.18,sale,8,True,63.992,low
8,Widget Max,WG-009,Widgets,59.99,0.42,bestseller,9,False,59.990,high
9,Doohickey Plus,DH-010,Doohickeys,24.99,0.20,new,10,False,24.990,low
